In [175]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns

In [157]:
load_dotenv()
API_KEY = os.getenv("APIKEY")
Channel_handeler = "@AJpluskibreet"

In [159]:
def ChannelNameToUploadeID(Channel_handeler):
    url = 'https://www.googleapis.com/youtube/v3/channels'

    param = {
        "part" : "contentDetails",
        "forHandle" : Channel_handeler,
        "key" : API_KEY
        
        }

    respons = requests.get(url,params=param)
    listId = respons.json()['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    return listId


In [160]:
uploadId = ChannelNameToUploadeID(Channel_handeler)

In [161]:
def VideoIdList(uploadId):
    url = 'https://www.googleapis.com/youtube/v3/playlistItems'
    token = 1
    Id_List=[]
    while token != None:
        if token == 1:
            token = None
        param = {
            "part" : "contentDetails",
            "playlistId" : uploadId,
            "maxResults" : 50,
            "key" : API_KEY,
            "pageToken" : token
        }

        respons = requests.get(url,params=param)
        plylst_data = respons.json()
        
        token = plylst_data.get("nextPageToken")
        for Index in plylst_data['items']:
            videoId = Index['contentDetails']['videoId']
            Id_List.append(videoId)
        
    return Id_List


In [144]:
Videos_Id_List = VideoIdList(uploadId)

In [ ]:
Videos_Id_List

In [ ]:
def videoDetails(Videos_Id_List):
    url = 'https://www.googleapis.com/youtube/v3/videos'
    list_items = {
                'snippet': ["publishedAt","title","tags"],
                'contentDetails' : ["duration","definition","caption","licensedContent"],
                'statistics' : ["viewCount","likeCount","commentCount"]
    }
    video_data_dict = {}
    row = []

    for name,feild in list_items.items():
        
        for i in feild :
            row.append(i)
            video_data_dict[i] = []

    video_cache = []

    # 1. Fetch all videos once
    for videoID in Videos_Id_List:
        params = {
            "part": "snippet,contentDetails,statistics",
            "id": videoID,
            "key": API_KEY
        }

        response = requests.get(url, params=params)
        data = response.json()

        if not data.get("items"):
            continue

        video_cache.append(data["items"][0])

    video_data_dict = {
        field: []
        for fields in list_items.values()
        for field in fields
    }

    for name, fields in list_items.items():
        for field in fields:

            col = []

            for vid in video_cache:
                value = vid.get(name, {}).get(field)
                col.append(value)

            video_data_dict[field] = col


    df = pd.DataFrame(video_data_dict)
    return df

In [ ]:
videoDetails(Videos_Id_List)

In [164]:
def Channel_Scraper(Channel_handeler,API_KEY):
    uploadId = ChannelNameToUploadeID(Channel_handeler)
    Videos_Id_List = VideoIdList(uploadId)
    DataFrame = videoDetails(Videos_Id_List)
    return DataFrame

In [168]:
Full_Channel_Data = Channel_Scraper(Channel_handeler,API_KEY)

In [170]:
Fcd = Full_Channel_Data

In [171]:
Fcd

,publishedAt,title,tags,categoryId,liveBroadcastContent,defaultLanguage,defaultAudioLanguage,duration,definition,caption,licensedContent,contentRating,projection,viewCount,likeCount,favoriteCount,commentCount
0,2026-04-29T17:00:01Z,الدحيح | عزيزي.. حزر فزر السلاحف بتتنفس منين؟,None,24,none,ar,ar,PT2M13S,hd,false,False,{},rectangular,17265,1340,0,40
1,2026-04-29T16:00:01Z,المُخبر الاقتصادي+ | لماذا نخسر في البورصة؟,None,24,none,ar,ar,PT1M33S,hd,false,False,{},rectangular,57466,2718,0,72
2,2026-04-28T18:00:42Z,المُخبر الاقتصادي+ | كيف تفوز الصين في حرب أمر...,"[Economy, إيران, الصين, المخبر الاقتصادي, جو ب...",24,none,ar,ar,PT25M48S,hd,false,True,{},rectangular,502946,13305,0,429
3,2026-04-28T17:00:00Z,الدحيح | السلاحف: أبطأ كائن وأطول نفس,None,24,none,ar,ar,PT2M11S,hd,false,False,{},rectangular,35097,1727,0,46
4,2026-04-27T18:00:06Z,الدحيح | كيف نجت السلاحف من الكوارث الطبيعية؟,"[الدحيح, AJ+ عربي, Eldahih, السليط, معنى و زيا...",24,none,ar,ar,PT34M16S,hd,true,True,{},rectangular,953564,36376,0,2161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4248,2015-11-29T17:24:08Z,الواقع المقلوب | +AJ عربي,None,25,none,ar,ar,PT15S,hd,false,True,{},rectangular,9911,330,0,12
4249,2015-11-29T06:00:00Z,الفرق في رواية القصة | +AJ عربي,"[aj plus, aj plus arabi, قصص قصيرة, قصص واقعية...",25,none,ar,ar,PT31S,hd,false,True,{},rectangular,20077,557,0,30
4250,2015-11-28T06:00:01Z,صوتك منبع إلهامنا | +AJ عربي,"[aj plus, aj plus arabi, قصص قصيرة, قصص واقعية...",25,none,ar,ar,PT12S,hd,false,True,{},rectangular,10884,361,0,15
4251,2015-11-27T06:00:01Z,قصصنا أحداث نعيشها معاً | +AJ عربي,"[aj plus, aj plus arabi, قصص قصيرة, قصص واقعية...",25,none,ar,ar,PT14S,hd,false,True,{},rectangular,33091,589,0,20


In [178]:
Fcd.to_json('AjplusData.json')